# A1组 6.3 实验：DeepONet 学习一维周期 Helmholtz 型算子

> 本 Notebook 只包含 6.3 节，配套正文为 `A1-G12-正文.md`。

- **问题：** 学习算子 $G:u\mapsto w$，其中 $(-w''+w)=u$，$y\in[0,1]$，周期边界；训练/验证/测试按输入函数划分，测试函数全部未见。
- **代码载体：** 全部代码在本 Notebook 内，按单元顺序执行；不导入外部 `.py`。
- **相对路径：** 从本文件所在的提交包根目录启动 Jupyter；图写入 `图片/`，数据写入 `数据/`，检查点写入 `辅助文件/`，不新建其他文件夹。
- **环境：** Python 3.12.15；PyTorch 2.14.1+cpu；NumPy 2.5.4；Matplotlib 3.11.2；CPU（4 线程）。
- **运行预算：** 数据生成约 0.2 s，主训练约 65–85 s（本次复跑 83.19 s），消融约 3 min（7 组共约 174 s）。正式长任务记录中的 807.83 s 含系统挂起，详见 `辅助文件/A1组-6.3-实验日志.md`。
- **产物：** `数据/A1组-6.3-主实验数据.npz`、`数据/A1组-6.3-主实验汇总.md`、`数据/A1组-6.3-消融结果.csv`、`图片/A1组-6.3-*.png`。


## 6.3 输入与独立参考

本节先固定数据生成协议：系数 $a_0\sim U(0.5,1.5)$，$a_k,b_k\sim U(-2/k,2/k)$，$K=4$；传感器点 $m=32$，输出网格 $M=64$。训练、验证、测试按输入函数索引互不相交；另设幅值放大 1.8 倍的 OOD 和最高谐波 $K=8$ 的频率 OOD。

数据标签由周期 Helmholtz 方程的解析解给出：
$$
u(y)=a_0+\sum_{k=1}^{K}\left[a_k\cos(2\pi ky)+b_k\sin(2\pi ky)\right],\qquad
w(y)=a_0+\sum_{k=1}^{K}\frac{a_k\cos(2\pi ky)+b_k\sin(2\pi ky)}{1+(2\pi k)^2}.
$$
谱求解器与二阶差分求解器只作为独立参考和成本基线，不参与训练。

In [1]:
import copy
import csv
import math
import os
import platform
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = Path("图片")
DATA = Path("数据")
AUX = Path("辅助文件")
assert FIG.is_dir() and DATA.is_dir() and AUX.is_dir(), (
    "缺少 图片/、数据/ 或 辅助文件/ 目录；请在提交包根目录启动 Jupyter"
)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


# =========================================================================
#                              配置
# =========================================================================
@dataclass
class Config:
    seed: int = 2024

    # --- 算子 / 数据 ---
    K: int = 4                 # 训练分布源项最高谐波次数
    M: int = 64                # 输出网格点数（训练与测试同一网格）
    m: int = 32                # 传感器点数
    ampScale: float = 1.0      # 训练分布系数幅值缩放
    oodAmpScale: float = 1.8   # 分布外(1)：系数幅值放大
    Kood: int = 8              # 分布外(2)：未见过的更高频率
    nRandomCoords: int = 16    # 每个测试函数的随机查询坐标数

    # --- 网络 ---
    p: int = 32                # 潜在维数
    hidden: int = 64           # 两个子网隐层宽度

    # --- 规模（按输入函数划分）---
    N_train: int = 1000
    N_val: int = 200
    N_test: int = 200

    # --- 优化 ---
    batchSize: int = 32
    maxEpochs: int = 1000      # 设计上限 5000；CPU 上默认 1000 + 早停
    lr: float = 1e-3
    lrDecay: bool = False      # True 时按几何衰减到 lrMin
    lrMin: float = 1e-4
    patience: int = 100        # 早停耐心（连续多少 epoch 无改善）

    # --- 运行控制 ---
    quick: bool = False        # True: 缩减规模冒烟测试
    runAblation: bool = False  # True: 运行消融实验
    saveResults: bool = True   # True: 保存 ndarray 结果 / 训练检查点
    ckptFile: str = str(AUX / "A1组-6.3-模型检查点.pt")
    resume: bool = False       # True: 从快照恢复，跳过训练直接评估
    figDir: str = str(FIG)

    ygrid: np.ndarray = field(init=False, repr=False)
    xsens: np.ndarray = field(init=False, repr=False)

    def __post_init__(self) -> None:
        self.refresh_grids()

    def refresh_grids(self) -> None:
        """周期网格 y_j = j/M, x_j = j/m（端点 1 不重复，适合 FFT）"""
        self.ygrid = np.arange(self.M) / self.M
        self.xsens = np.arange(self.m) / self.m
        assert self.M % 2 == 0, "M 必须为偶数（谱求解器与谱微分都用 FFT）"


In [2]:
def sample_coeffs(rng: np.random.Generator, N: int, K: int, amp: float) -> dict:
    a0 = 0.5 + rng.random(N)
    s = 2.0 * amp / np.arange(1, K + 1)          # 与设计一致: |a_k| <= 2*amp/k
    a = rng.uniform(-1.0, 1.0, size=(N, K)) * s
    b = rng.uniform(-1.0, 1.0, size=(N, K)) * s
    return {"a0": a0, "a": a, "b": b}


def eval_u(C: dict, x: np.ndarray) -> np.ndarray:
    """u 在坐标 x 上的取值 -> (N, len(x))"""
    K = C["a"].shape[1]
    x = np.asarray(x, dtype=np.float64).ravel()
    U = np.repeat(C["a0"][:, None], x.size, axis=1)
    for k in range(1, K + 1):
        U = U + C["a"][:, k - 1][:, None] * np.cos(2 * np.pi * k * x) \
              + C["b"][:, k - 1][:, None] * np.sin(2 * np.pi * k * x)
    return U


def eval_w_exact(C: dict, y: np.ndarray) -> np.ndarray:
    """解析解（网格形式），仅用于报告求解器误差 -> (N, len(y))"""
    K = C["a"].shape[1]
    y = np.asarray(y, dtype=np.float64).ravel()
    W = np.repeat(C["a0"][:, None], y.size, axis=1)
    for k in range(1, K + 1):
        d = 1.0 + (2 * np.pi * k) ** 2
        W = W + (C["a"][:, k - 1][:, None] * np.cos(2 * np.pi * k * y)
                 + C["b"][:, k - 1][:, None] * np.sin(2 * np.pi * k * y)) / d
    return W


def eval_w_exact_coords(C: dict, Y: np.ndarray) -> np.ndarray:
    """解析解（每个函数有自己的查询坐标）
    Y: Q x N（第 n 列是第 n 个函数的 Q 个坐标） -> W: Q x N
    """
    Q, N = Y.shape
    K = C["a"].shape[1]
    W = np.repeat(C["a0"][None, :], Q, axis=0)
    for k in range(1, K + 1):
        d = 1.0 + (2 * np.pi * k) ** 2
        W = W + (C["a"][:, k - 1][None, :] * np.cos(2 * np.pi * k * Y)
                 + C["b"][:, k - 1][None, :] * np.sin(2 * np.pi * k * Y)) / d
    return W


def subset_coef(C: dict, i: int) -> dict:
    return {"a0": np.array([C["a0"][i]]), "a": C["a"][i:i + 1, :], "b": C["b"][i:i + 1, :]}


# ---------------- 传统求解器（基线）----------------
def solve_spectral(u: np.ndarray, M: int) -> np.ndarray:
    """解 (-w''+w)=u，周期边界，谱方法：w_hat = u_hat / (1+(2*pi*k)^2)
    对 K < M/2 的带限右端是机器精度精确的。
    """
    u = np.asarray(u, dtype=np.float64).ravel()
    kk = np.concatenate([np.arange(0, M // 2), np.arange(-M // 2, 0)])
    lam = 1.0 + (2 * np.pi * kk) ** 2
    return np.real(np.fft.ifft(np.fft.fft(u) / lam))


def solve_fd(u: np.ndarray, M: int) -> np.ndarray:
    """同一问题的二阶中心差分 + 周期边界（通用求解器参照，误差 O(h^2)）"""
    u = np.asarray(u, dtype=np.float64).ravel()
    h = 1.0 / M
    off = -1.0 / h ** 2
    main = 2.0 / h ** 2 + 1.0
    A = np.zeros((M, M), dtype=np.float64)
    np.fill_diagonal(A, main)
    idx = np.arange(M - 1)
    A[idx, idx + 1] = off
    A[idx + 1, idx] = off
    A[0, M - 1] = off
    A[M - 1, 0] = off
    return np.linalg.solve(A, u)


# ---------------- 数据集 ----------------
def make_split(rng: np.random.Generator, N: int, m: int, M: int, K: int,
               amp: float, x: np.ndarray, y: np.ndarray) -> dict:
    """一个 split = N 个独立输入函数，标签由谱求解器给出"""
    C = sample_coeffs(rng, N, K, amp)
    u = eval_u(C, x)            # N x m  传感器读数（branch 输入）
    Uy = eval_u(C, y)           # N x M  同一函数在输出网格上的值（求解器右端）
    w = np.zeros((N, M), dtype=np.float64)
    for i in range(N):
        w[i, :] = solve_spectral(Uy[i, :], M)   # 一次求解器调用 = 一个函数
    return {"C": C, "u": u, "w": w, "randY": None}


def make_dataset(cfg: Config, rng: np.random.Generator) -> dict:
    x, y = cfg.xsens, cfg.ygrid
    data = {
        "tr":   make_split(rng, cfg.N_train, cfg.m, cfg.M, cfg.K, cfg.ampScale, x, y),
        "va":   make_split(rng, cfg.N_val, cfg.m, cfg.M, cfg.K, cfg.ampScale, x, y),
        "te":   make_split(rng, cfg.N_test, cfg.m, cfg.M, cfg.K, cfg.ampScale, x, y),
        "ood":  make_split(rng, cfg.N_test, cfg.m, cfg.M, cfg.K, cfg.oodAmpScale, x, y),
        "oodf": make_split(rng, cfg.N_test, cfg.m, cfg.M, cfg.Kood, cfg.ampScale, x, y),
    }
    # 每个函数一次求解器调用
    data["nCalls"] = cfg.N_train + cfg.N_val + 3 * cfg.N_test

    # 为随机坐标测试生成“每函数各不相同”的 16 个查询点（固定在测试集上，只生成一次）
    Q = cfg.nRandomCoords
    data["te"]["randY"] = rng.random((Q, cfg.N_test))
    data["ood"]["randY"] = rng.random((Q, cfg.N_test))
    data["oodf"]["randY"] = rng.random((Q, cfg.N_test))
    return data


In [3]:
print("==================== DeepONet 算子学习 ====================")
cfg = Config()
rng = np.random.default_rng(cfg.seed)
torch.manual_seed(cfg.seed)
torch.set_num_threads(min(4, os.cpu_count() or 1))

print("---- 1. 数据生成 ----")
t_gen = time.perf_counter()
data = make_dataset(cfg, rng)
gen_wall = time.perf_counter() - t_gen

print(f"  训练 / 验证 / 测试 函数数: {cfg.N_train} / {cfg.N_val} / {cfg.N_test}  （索引互不相交）")
print(f"  传感器点 m={cfg.m}, 输出网格 M={cfg.M}")
print(f"  额外集合: 幅值x{cfg.oodAmpScale:g} 的 OOD {cfg.N_test} 个；"
      f"K={cfg.Kood} 的高频 OOD {cfg.N_test} 个")
print(f"  求解器调用总次数: {data['nCalls']} "
      f"（其中主集合 {cfg.N_train + cfg.N_val + cfg.N_test} 次 = "
      f"{cfg.N_train}+{cfg.N_val}+{cfg.N_test}）")
print(f"  数据生成总墙钟: {gen_wall:.2f} s")

n_w = sum(data[k]["w"].size for k in ("tr", "va", "te"))
n_u = sum(data[k]["u"].size for k in ("tr", "va", "te"))
print(f"  标签存储: {8 * n_w / 1024:.1f} KB ；传感器输入存储: {8 * n_u / 1024:.1f} KB")


==================== DeepONet 算子学习 ====================
---- 1. 数据生成 ----
  训练 / 验证 / 测试 函数数: 1000 / 200 / 200  （索引互不相交）
  传感器点 m=32, 输出网格 M=64
  额外集合: 幅值x1.8 的 OOD 200 个；K=8 的高频 OOD 200 个
  求解器调用总次数: 1800 （其中主集合 1400 次 = 1000+200+200）
  数据生成总墙钟: 0.04 s
  标签存储: 700.0 KB ；传感器输入存储: 350.0 KB


## 6.3 方法实现与运行

DeepONet 由 branch 与 trunk 两个 tanh MLP 组成：branch 编码 $m=32$ 个传感器值，trunk 编码查询坐标 $y$，二者在 $p=32$ 维潜在空间做内积：
$$
G_\theta(u)(y)=\sum_{j=1}^{p}B_j(u)\,T_j(y)+b_0.
$$
训练用 MSE、Adam（学习率 $10^{-3}$）、batch 32、最多 1000 epoch，并以验证集相对 $L^2$ 误差早停（耐心 100）。所有随机种子固定为 2024。

In [4]:
class MLP(nn.Module):
    """branch / trunk 都是 2 隐层 tanh MLP（Glorot 初始化，与原 MATLAB 一致）"""

    def __init__(self, n_in: int, n_out: int, hidden: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, n_out),
        )
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DeepONet(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.branch = MLP(cfg.m, cfg.p, cfg.hidden)
        self.trunk = MLP(1, cfg.p, cfg.hidden)
        self.b0 = nn.Parameter(torch.zeros(1))     # 内积后的标量偏置

    def forward(self, uB: torch.Tensor, yM: torch.Tensor) -> torch.Tensor:
        """uB: B x m ; yM: M x 1 -> w_pred: M x B"""
        b = self.branch(uB)                        # B x p
        t = self.trunk(yM)                         # M x p
        return t @ b.T + self.b0                   # M x B


@torch.no_grad()
def predict_operator(model: DeepONet, U: np.ndarray, y: np.ndarray) -> np.ndarray:
    """U: m x N (double)   y: M   ->   W: M x N (double)"""
    was_training = model.training
    model.eval()
    uT = torch.as_tensor(np.asarray(U, dtype=np.float32).T)          # N x m
    yT = torch.as_tensor(np.asarray(y, dtype=np.float32).reshape(-1, 1))  # M x 1
    b = model.branch(uT)                                             # N x p
    t = model.trunk(yT)                                              # M x p
    W = (t @ b.T + model.b0).numpy().astype(np.float64)
    if was_training:
        model.train()
    return W


@torch.no_grad()
def predict_operator_coords(model: DeepONet, U: np.ndarray, Y: np.ndarray) -> np.ndarray:
    """U: m x N   Y: Q x N（每个函数自己的查询坐标）  ->  W: Q x N"""
    was_training = model.training
    model.eval()
    N = U.shape[1]
    Q = Y.shape[0]
    uT = torch.as_tensor(np.asarray(U, dtype=np.float32).T)          # N x m
    # 按 n 优先、q 次之展平，保证与 (Q, N) 的列对应关系
    Yf = np.asarray(Y, dtype=np.float32).T.reshape(-1, 1)            # (N*Q) x 1
    b = model.branch(uT)                                             # N x p
    t = model.trunk(torch.as_tensor(Yf)).reshape(N, Q, -1).permute(1, 0, 2)  # Q x N x p
    W = (t * b.unsqueeze(0)).sum(dim=-1) + model.b0                  # Q x N
    W = W.numpy().astype(np.float64)
    if was_training:
        model.train()
    return W


def rel_l2_cols(Wp: np.ndarray, Wt: np.ndarray) -> np.ndarray:
    """逐函数（逐列）相对 L2；均匀网格上 sqrt(mean(.)) 的 1/M 在比值里抵消"""
    return np.sqrt(np.sum((Wp - Wt) ** 2, axis=0)) / np.sqrt(np.sum(Wt ** 2, axis=0))


def spec_deriv(W: np.ndarray, M: int) -> np.ndarray:
    """周期网格上的谱微分（沿第 0 维）"""
    kk = np.concatenate([np.arange(0, M // 2), np.arange(-M // 2, 0)])
    return np.real(np.fft.ifft(1j * 2 * np.pi * kk[:, None] * np.fft.fft(W, axis=0), axis=0))


def train_deeponet(cfg: Config, data: dict, rng: np.random.Generator) -> tuple[DeepONet, dict]:
    model = DeepONet(cfg)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)   # betas/eps 与 adamupdate 默认一致

    yTr = torch.as_tensor(cfg.ygrid, dtype=torch.float32).reshape(-1, 1)   # M x 1
    uVa = torch.as_tensor(data["va"]["u"], dtype=torch.float32)            # N x m
    wVa = data["va"]["w"].T                                                # M x N

    Ntr = data["tr"]["u"].shape[0]
    nb = Ntr // cfg.batchSize
    best = {"val": float("inf"), "epoch": 0, "state": None}
    wait = 0
    stopped_early = False

    hist = {"trainMSE": [], "valRel": [], "lr": []}
    t_start = time.perf_counter()
    epoch = 0

    for epoch in range(1, cfg.maxEpochs + 1):
        if cfg.lrDecay:
            frac = (epoch - 1) / max(1, cfg.maxEpochs - 1)
            lr = cfg.lr * (cfg.lrMin / cfg.lr) ** frac
        else:
            lr = cfg.lr
        for group in opt.param_groups:
            group["lr"] = lr

        perm = rng.permutation(Ntr)
        uE = data["tr"]["u"][perm]
        wE = data["tr"]["w"][perm]

        mse_sum = 0.0
        for ib in range(nb):
            rows = slice(ib * cfg.batchSize, (ib + 1) * cfg.batchSize)
            uB = torch.as_tensor(uE[rows], dtype=torch.float32)          # B x m
            wB = torch.as_tensor(wE[rows].T, dtype=torch.float32)        # M x B

            w_pred = model(uB, yTr)
            loss = ((w_pred - wB) ** 2).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            mse_sum += float(loss.detach())

        # --- 验证集：函数级相对 L2 误差（早停依据）---
        model.eval()
        with torch.no_grad():
            Wva = (model.trunk(yTr) @ model.branch(uVa).T + model.b0).numpy().astype(np.float64)
        model.train()
        val_rel = float(np.mean(rel_l2_cols(Wva, wVa)))

        hist["trainMSE"].append(mse_sum / max(nb, 1))
        hist["valRel"].append(val_rel)
        hist["lr"].append(lr)

        if val_rel < best["val"] * (1 - 1e-4):
            best["val"] = val_rel
            best["epoch"] = epoch
            best["state"] = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % max(1, round(cfg.maxEpochs / 10)) == 0:
            print(f"  epoch {epoch:4d} | trainMSE {hist['trainMSE'][-1]:.3e} | "
                  f"val rel-L2 {val_rel:.3e} | lr {lr:.1e} | "
                  f"{time.perf_counter() - t_start:6.1f} s", flush=True)

        if wait >= cfg.patience:
            stopped_early = True
            break

        # 长任务保护：定期存快照，崩溃/中断后可断点续跑
        if cfg.saveResults and epoch % 100 == 0:
            torch.save({
                "model": model.state_dict(),
                "hist": {k: list(v) for k, v in hist.items()},
                "epoch": epoch,
                "wallSoFar": time.perf_counter() - t_start,
            }, cfg.ckptFile)

    wall_total = time.perf_counter() - t_start

    # 恢复验证集最优权重
    if best["state"] is not None:
        model.load_state_dict(best["state"])

    hist["wallTotal"] = wall_total
    hist["epochsRun"] = epoch
    hist["bestEpoch"] = best["epoch"]
    hist["bestVal"] = best["val"]
    hist["stoppedEarly"] = stopped_early
    return model, hist


# =========================================================================
#                          评估


In [5]:
print("\n---- 2. 训练（Adam + 验证集早停）----")
if cfg.resume and os.path.isfile(cfg.ckptFile):
    ckpt = torch.load(cfg.ckptFile, weights_only=False)
    model = DeepONet(cfg)
    model.load_state_dict(ckpt["model"])
    hist = ckpt["hist"]
    hist["wallTotal"] = ckpt["wallSoFar"]
    hist["epochsRun"] = ckpt["epoch"]
    print(f"  从检查点恢复（已训练 {ckpt['epoch']} 个 epoch），跳过训练")
else:
    model, hist = train_deeponet(cfg, data, rng)
    if cfg.saveResults:
        torch.save({"model": model.state_dict(), "hist": hist,
                    "epoch": hist["epochsRun"], "wallSoFar": hist["wallTotal"]}, cfg.ckptFile)
        print(f"  训练检查点已保存: {cfg.ckptFile}")
print(f"  训练结束: 运行 {hist['epochsRun']} 个 epoch（最好 epoch={hist['bestEpoch']}, "
      f"val rel-L2={hist['bestVal']:.3e}），早停={hist['stoppedEarly']}")
print(f"  训练总墙钟: {hist['wallTotal']:.2f} s")



---- 2. 训练（Adam + 验证集早停）----
  epoch    1 | trainMSE 2.777e-01 | val rel-L2 2.652e-01 | lr 1.0e-03 |    0.1 s
  epoch  100 | trainMSE 9.701e-04 | val rel-L2 3.323e-02 | lr 1.0e-03 |    6.5 s
  epoch  200 | trainMSE 3.346e-04 | val rel-L2 1.953e-02 | lr 1.0e-03 |   13.0 s
  epoch  300 | trainMSE 3.055e-04 | val rel-L2 3.321e-02 | lr 1.0e-03 |   19.0 s
  epoch  400 | trainMSE 2.953e-04 | val rel-L2 1.571e-02 | lr 1.0e-03 |   24.9 s
  epoch  500 | trainMSE 7.543e-05 | val rel-L2 7.712e-03 | lr 1.0e-03 |   34.8 s
  epoch  600 | trainMSE 1.102e-04 | val rel-L2 7.180e-03 | lr 1.0e-03 |   45.1 s
  epoch  700 | trainMSE 5.275e-05 | val rel-L2 7.943e-03 | lr 1.0e-03 |   55.8 s
  epoch  800 | trainMSE 2.093e-05 | val rel-L2 5.961e-03 | lr 1.0e-03 |   66.4 s
  epoch  900 | trainMSE 4.279e-04 | val rel-L2 9.246e-03 | lr 1.0e-03 |   77.8 s
  训练结束: 运行 948 个 epoch（最好 epoch=848, val rel-L2=4.176e-03），早停=True
  训练总墙钟: 83.19 s
  训练检查点已保存: 辅助文件\A1组-6.3-模型检查点.pt


## 6.3 误差图表与对比

评价指标为逐函数相对 $L^2$ 误差、中位数、P95、最大绝对误差和 $H^1$ 相对误差。除测试集外，还报告：

- **同函数新坐标**：同一批测试函数、每个函数 16 个随机查询坐标，用于检查 trunk 的坐标泛化；
- **训练函数**：只作参考，不能作为算子学习证据；
- **记忆基线**：在传感器空间找最近训练函数并直接搬标签，用于判断模型是否只是“记住”训练函数；
- **OOD**：幅值放大与更高频率两组分布外输入。

图 6.3-1 至 6.3-4 分别对应训练曲线、测试误差分布、预测示例和分布内外对比。

In [6]:
def local_pct(x: np.ndarray, pct: float) -> float:
    """不依赖额外库的分位数（线性插值，与 MATLAB 版一致）"""
    return float(np.percentile(np.asarray(x).ravel(), pct))


def _summarize(eL: np.ndarray, max_abs: float, mean_h1: float) -> dict:
    return {
        "eL2": eL,
        "mean": float(np.mean(eL)),
        "median": float(np.median(eL)),
        "p95": local_pct(eL, 95),
        "maxRel": float(np.max(eL)),
        "maxAbs": float(max_abs),
        "meanH1": mean_h1,
    }


def evaluate_split(model: DeepONet, split: dict, cfg: Config) -> dict:
    """在主输出网格（64 个等距点）上评估，真值用求解器标签 G_lab"""
    Wtrue = split["w"].T                                   # M x N
    Wpred = predict_operator(model, split["u"].T, cfg.ygrid)

    E = Wpred - Wtrue
    eL = rel_l2_cols(Wpred, Wtrue)

    M = cfg.ygrid.size
    dE = spec_deriv(E, M)
    dW = spec_deriv(Wtrue, M)
    eH = np.sqrt(np.sum(E ** 2, axis=0) + np.sum(dE ** 2, axis=0)) \
        / np.sqrt(np.sum(Wtrue ** 2, axis=0) + np.sum(dW ** 2, axis=0))

    R = _summarize(eL, float(np.max(np.abs(E))), float(np.mean(eH)))
    R["Wpred"] = Wpred
    R["Wtrue"] = Wtrue
    return R


def evaluate_random_coords(model: DeepONet, split: dict, cfg: Config) -> dict:
    """同一批函数、每个函数另取 16 个随机坐标（“同一函数的新查询坐标”）"""
    Y = split["randY"]                                     # Q x N
    Wtrue = eval_w_exact_coords(split["C"], Y)             # Q x N（解析解，求解器无网格定义）
    Wpred = predict_operator_coords(model, split["u"].T, Y)

    E = Wpred - Wtrue
    eL = rel_l2_cols(Wpred, Wtrue)

    R = _summarize(eL, float(np.max(np.abs(E))), float("nan"))
    R["Wpred"] = Wpred
    R["Wtrue"] = Wtrue
    return R


def memorization_baseline(train_split: dict, test_split: dict) -> dict:
    """“只记住训练函数”基线：在传感器空间找最近的训练函数，直接搬它的标签"""
    Utr = train_split["u"]
    Ute = test_split["u"]
    d = ((Ute[:, None, :] - Utr[None, :, :]) ** 2).sum(axis=-1)   # Nte x Ntr
    nn_idx = np.argmin(d, axis=1)

    Wpred = train_split["w"][nn_idx].T                            # M x Nte
    Wtrue = test_split["w"].T
    eL = rel_l2_cols(Wpred, Wtrue)

    R = _summarize(eL, float(np.max(np.abs(Wpred - Wtrue))), float("nan"))
    R["Wpred"] = Wpred
    R["Wtrue"] = Wtrue
    return R


def print_metrics(name: str, R: dict) -> None:
    h1 = "  --  " if math.isnan(R["meanH1"]) else f"{R['meanH1']:.3e}"
    print(f"  {name:<34s} 均值 {R['mean']:.3e} | 中位数 {R['median']:.3e} | "
          f"P95 {R['p95']:.3e} | 最大 {R['maxRel']:.3e} | "
          f"最大绝对误差 {R['maxAbs']:.3e} | H1均值 {h1}", flush=True)


# =========================================================================
#                          成本 / 系统信息


In [7]:
print("\n---- 3. 测试集评估（全部是新输入函数）----")
Rg = evaluate_split(model, data["te"], cfg)            # 64 等距点
Rr = evaluate_random_coords(model, data["te"], cfg)    # 每函数 16 个随机坐标
Rt = evaluate_split(model, data["tr"], cfg)            # 训练函数（仅参考）
Rm = memorization_baseline(data["tr"], data["te"])     # “只记住训练函数”基线
Ro = evaluate_split(model, data["ood"], cfg)           # 幅值 OOD
Rf = evaluate_split(model, data["oodf"], cfg)          # 频率 OOD

print_metrics("测试集(新函数, 64网格点)", Rg)
print_metrics("测试集(新函数, 16随机坐标)", Rr)
print_metrics("训练函数(仅参考, 不可作证据)", Rt)
print_metrics("最近邻“记忆”基线", Rm)
print_metrics(f"OOD 幅值 x{cfg.oodAmpScale:g}", Ro)
print_metrics(f"OOD 频率 K={cfg.Kood}", Rf)

WexTe = eval_w_exact(data["te"]["C"], cfg.ygrid)
e_sol = rel_l2_cols(data["te"]["w"].T, WexTe.T)
print(f"  求解器自身相对解析解误差: 均值 {np.mean(e_sol):.2e} | 最大 {np.max(e_sol):.2e}")

print("\n---- 5. 判据 ----")
ratio = Rg["mean"] / Rm["mean"]
print(f"  新函数误差 / 记忆基线误差 = {ratio:.3f}")
if Rg["mean"] < 0.5 * Rm["mean"]:
    print("  -> 新输入函数上的误差显著低于“只记住训练函数”的基线，支持“学到了算子”。")
else:
    print("  -> 与记忆基线差距不足，不能宣称学到了算子。")
print(f"  新函数(网格) vs 同函数新坐标: {Rg['mean']:.3e} vs {Rr['mean']:.3e} —— "
      "二者接近说明 trunk 在坐标方向确实泛化；但只有“新函数”上的误差才是算子学习的证据。")



---- 3. 测试集评估（全部是新输入函数）----
  测试集(新函数, 64网格点)                    均值 4.144e-03 | 中位数 3.857e-03 | P95 7.131e-03 | 最大 1.041e-02 | 最大绝对误差 1.825e-02 | H1均值 9.257e-02
  测试集(新函数, 16随机坐标)                   均值 3.983e-03 | 中位数 3.664e-03 | P95 7.192e-03 | 最大 1.190e-02 | 最大绝对误差 1.631e-02 | H1均值   --  
  训练函数(仅参考, 不可作证据)                   均值 4.090e-03 | 中位数 3.685e-03 | P95 7.808e-03 | 最大 1.024e-02 | 最大绝对误差 1.957e-02 | H1均值 9.620e-02
  最近邻“记忆”基线                          均值 1.523e-01 | 中位数 1.223e-01 | P95 3.755e-01 | 最大 6.686e-01 | 最大绝对误差 5.090e-01 | H1均值   --  
  OOD 幅值 x1.8                        均值 8.813e-03 | 中位数 8.672e-03 | P95 1.472e-02 | 最大 2.361e-02 | 最大绝对误差 3.356e-02 | H1均值 1.681e-01
  OOD 频率 K=8                         均值 1.194e-01 | 中位数 8.906e-02 | P95 3.208e-01 | 最大 5.781e-01 | 最大绝对误差 4.892e-01 | H1均值 5.281e-01
  求解器自身相对解析解误差: 均值 1.29e-16 | 最大 2.87e-16

---- 5. 判据 ----
  新函数误差 / 记忆基线误差 = 0.027
  -> 新输入函数上的误差显著低于“只记住训练函数”的基线，支持“学到了算子”。
  新函数(网格) vs 同函数新坐标: 4.144e-03 vs 3.983e-03 —— 二者接近说明

In [8]:
def timeit_median(fn, warmup: int = 3, repeats: int = 20) -> float:
    """对应 MATLAB timeit：多次测量取中位数（含预热）"""
    for _ in range(warmup):
        fn()
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


def amortization(Ttrain: float, t_solver: float, t_net: float) -> tuple[str, float]:
    den = t_solver - t_net
    if den > 0:
        n = Ttrain / den
        msg = (f"存在正的摊销点，需 {n:.3g} 次推理 "
               f"(训练 {Ttrain:.3g} s / 每次净节省 {den * 1e3:.3g} ms)")
    else:
        n = float("inf")
        msg = (f"不存在正的摊销点：求解器单次 {t_solver * 1e3:.3f} ms "
               f"不慢于网络单次 {t_net * 1e3:.3f} ms")
    return msg, n


def hw_info() -> dict:
    return {
        "os": f"{platform.system()} {platform.release()}",
        "cpu": os.environ.get("PROCESSOR_IDENTIFIER", platform.processor() or "unknown"),
        "cores": os.cpu_count(),
        "rel": f"Python {platform.python_version()} / torch {torch.__version__}",
    }


In [9]:
print("\n---- 4. 成本记录（全部为实测）----")
hw = hw_info()
print(f"  平台: {hw['os']} | {hw['cpu']} | 逻辑核 {hw['cores']} | {hw['rel']}")

u_one = data["te"]["u"][0]                              # m
t_net = timeit_median(lambda: predict_operator(model, u_one.reshape(-1, 1), cfg.ygrid))

C1 = subset_coef(data["te"]["C"], 0)
u_grid = eval_u(C1, cfg.ygrid).T                        # M x 1
t_spec = timeit_median(lambda: solve_spectral(u_grid, cfg.M))
t_fd = timeit_median(lambda: solve_fd(u_grid, cfg.M))

e_spec = np.mean(rel_l2_cols(solve_spectral(u_grid, cfg.M).reshape(-1, 1),
                            eval_w_exact(C1, cfg.ygrid).T))
e_fd = np.mean(rel_l2_cols(solve_fd(u_grid, cfg.M).reshape(-1, 1),
                           eval_w_exact(C1, cfg.ygrid).T))

print(f"  网络单次推理 (M={cfg.M}): {t_net * 1e3:.3f} ms")
print(f"  谱求解器单次     : {t_spec * 1e3:.3f} ms   （自身相对误差 {e_spec:.1e}）")
print(f"  二阶差分求解器单次: {t_fd * 1e3:.3f} ms   （自身相对误差 {e_fd:.1e}）")

T_train = hist["wallTotal"]
print("  摊销点（训练成本 / (求解器单次 - 网络单次)）:")
msg_spec, n_spec = amortization(T_train, t_spec, t_net)
msg_fd, n_fd = amortization(T_train, t_fd, t_net)
print(f"    对谱求解器  : {msg_spec}")
print(f"    对差分求解器: {msg_fd}")
if math.isfinite(n_spec):
    print(f"    （谱求解器需要约 {n_spec:.3g} 次新函数推理才能摊平训练成本）")
if math.isfinite(n_fd):
    print(f"    （差分求解器需要约 {n_fd:.3g} 次新函数推理才能摊平训练成本）")



---- 4. 成本记录（全部为实测）----
  平台: Windows 11 | Intel64 Family 6 Model 140 Stepping 2, GenuineIntel | 逻辑核 8 | Python 3.12.15 / torch 2.14.1+cpu
  网络单次推理 (M=64): 0.307 ms
  谱求解器单次     : 0.040 ms   （自身相对误差 1.7e-16）
  二阶差分求解器单次: 0.046 ms   （自身相对误差 3.1e-05）
  摊销点（训练成本 / (求解器单次 - 网络单次)）:
    对谱求解器  : 不存在正的摊销点：求解器单次 0.040 ms 不慢于网络单次 0.307 ms
    对差分求解器: 不存在正的摊销点：求解器单次 0.046 ms 不慢于网络单次 0.307 ms


In [10]:
def save_figure(fig, name: str, cfg: Config) -> None:
    assert os.path.isdir(cfg.figDir), "缺少 图片/ 目录"
    fname = os.path.join(cfg.figDir, name)
    fig.savefig(fname + ".png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  图已保存: {fname}.png", flush=True)


def _clean_axes(ax) -> None:
    ax.grid(alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def make_figures(cfg: Config, data: dict, hist: dict, Rg: dict, Rr: dict,
                 Rt: dict, Rm: dict, Ro: dict, Rf: dict) -> None:
    y = cfg.ygrid

    # ---- 图 1: 训练曲线 ----
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].semilogy(hist["trainMSE"], lw=1.4)
    _clean_axes(axes[0])
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("训练 MSE")
    axes[0].set_title("(a) 训练损失")

    axes[1].semilogy(hist["valRel"], lw=1.4)
    ylo, yhi = axes[1].get_ylim()
    axes[1].plot([hist["bestEpoch"]] * 2, [ylo, yhi], "r--", lw=1.0)
    axes[1].plot(hist["bestEpoch"], hist["bestVal"], "ro", ms=6)
    _clean_axes(axes[1])
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("验证集相对 $L^2$ 误差")
    axes[1].set_title(f"(b) 验证集：最优 {hist['bestVal']:.2e} @ epoch {hist['bestEpoch']}")
    fig.tight_layout()
    save_figure(fig, "A1组-6.3-训练曲线", cfg)

    # ---- 图 2: 两个预测示例（分布内 / 频率 OOD）----
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    ax = axes[0]
    ax.plot(y, Rg["Wtrue"][:, 0], "b-", lw=1.8, label="真解 $G_{lab}$")
    ax.plot(y, Rg["Wpred"][:, 0], "r--", lw=1.8, label="DeepONet")
    Yr = data["te"]["randY"][:, 0]
    Wr = eval_w_exact_coords(subset_coef(data["te"]["C"], 0), Yr.reshape(1, -1))[0]
    ax.plot(Yr, Wr, "ko", mfc="y", ms=5, label="16 个随机坐标")
    _clean_axes(ax)
    ax.legend(loc="best")
    ax.set_xlabel("y")
    ax.set_ylabel("w(y)")
    ax.set_title(f"(a) 分布内测试函数 #1：相对误差 {Rg['eL2'][0]:.2e}")

    ax = axes[1]
    ax.plot(y, Rf["Wtrue"][:, 0], "b-", lw=1.8, label="真解")
    ax.plot(y, Rf["Wpred"][:, 0], "r--", lw=1.8, label="DeepONet")
    _clean_axes(ax)
    ax.legend(loc="best")
    ax.set_xlabel("y")
    ax.set_ylabel("w(y)")
    ax.set_title(f"(b) 频率 OOD（K={cfg.Kood}）#1：相对误差 {Rf['eL2'][0]:.2e}")
    fig.tight_layout()
    save_figure(fig, "A1组-6.3-预测示例", cfg)

    # ---- 图 3: 误差分布 ----
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    axes[0].hist(np.log10(Rg["eL2"]), bins=14)
    _clean_axes(axes[0])
    axes[0].set_xlabel(r"$\log_{10}$(相对 $L^2$ 误差)")
    axes[0].set_ylabel("函数个数")
    axes[0].set_title(f"(a) {cfg.N_test} 个测试函数的误差直方图")

    es = np.sort(Rg["eL2"])[::-1] * 1e3
    axes[1].plot(np.arange(1, es.size + 1), es, "-o", ms=3, lw=1.2, label="逐函数误差")
    axes[1].axhline(Rg["median"] * 1e3, color="g", ls="--", lw=1.2, label="中位数")
    axes[1].axhline(Rg["p95"] * 1e3, color="m", ls="--", lw=1.2, label="P95")
    _clean_axes(axes[1])
    axes[1].legend(loc="upper right")
    axes[1].set_xlabel("函数序号（按误差降序）")
    axes[1].set_ylabel(r"相对 $L^2$ 误差 ($\times10^{-3}$)")
    axes[1].set_title("(b) 误差排序曲线")
    fig.tight_layout()
    save_figure(fig, "A1组-6.3-测试误差分布", cfg)

    # ---- 图 4: 分布内 / OOD / 记忆基线 ----
    names = ["分布内", f"OOD 幅值x{cfg.oodAmpScale}", f"OOD K={cfg.Kood}", "记忆基线"]
    mu_all = np.array([Rg["mean"], Ro["mean"], Rf["mean"], Rm["mean"]])
    md_all = np.array([Rg["median"], Ro["median"], Rf["median"], Rm["median"]])

    fig, ax = plt.subplots(figsize=(7.8, 4.2))
    xpos = np.arange(1, 5)
    w = 0.32
    ax.bar(xpos - w / 2 - 0.01, mu_all, width=w, label="均值")
    ax.bar(xpos + w / 2 + 0.01, md_all, width=w, label="中位数")
    ax.set_yscale("log")
    ax.set_ylim(5e-4, mu_all.max() * 3)
    ax.set_xticks(xpos)
    ax.set_xticklabels(names)
    _clean_axes(ax)
    ax.set_ylabel("相对 $L^2$ 误差")
    ax.legend(loc="upper left")
    ax.set_title('(4) 分布内 vs 分布外 vs “记住训练函数”基线')
    for i in range(4):
        ax.text(i + 1 - w / 2 - 0.01, mu_all[i] * 1.2, f"{mu_all[i]:.1e}",
                ha="center", fontsize=8)
        ax.text(i + 1 + w / 2 + 0.01, md_all[i] * 1.2, f"{md_all[i]:.1e}",
                ha="center", fontsize=8)
    fig.tight_layout()
    save_figure(fig, "A1组-6.3-分布内外对比", cfg)


In [11]:
print("\n---- 6. 可视化 ----")
make_figures(cfg, data, hist, Rg, Rr, Rt, Rm, Ro, Rf)
print("  四张图已写入 图片/：")
for nm in ("A1组-6.3-训练曲线", "A1组-6.3-测试误差分布",
           "A1组-6.3-预测示例", "A1组-6.3-分布内外对比"):
    print(f"    图片/{nm}.png")



---- 6. 可视化 ----
  图已保存: 图片\A1组-6.3-训练曲线.png
  图已保存: 图片\A1组-6.3-预测示例.png
  图已保存: 图片\A1组-6.3-测试误差分布.png
  图已保存: 图片\A1组-6.3-分布内外对比.png
  四张图已写入 图片/：
    图片/A1组-6.3-训练曲线.png
    图片/A1组-6.3-测试误差分布.png
    图片/A1组-6.3-预测示例.png
    图片/A1组-6.3-分布内外对比.png


## 6.3 消融实验

为控制 CPU 时间，消融统一用 250 epoch + 早停、$N_{\rm test}=100$，每组从同一随机流开始。比较 $m=8/16/32$、$p=16/32/64$、$N_{\rm train}=100/300/1000$ 七组配置。结果写入 `数据/A1组-6.3-消融结果.csv`。

In [12]:
def run_ablation(base_cfg: Config) -> list[dict]:
    from dataclasses import replace

    print("\n---- 7. 消融实验（m / p / N_train）----")
    print("  说明：为控制 CPU 时间，消融统一用 250 epoch + 早停、N_test=100。")

    specs = [
        ("baseline m32 p32 N1000", {}),
        ("m=8",                    {"m": 8}),
        ("m=16",                   {"m": 16}),
        ("p=16",                   {"p": 16}),
        ("p=64",                   {"p": 64}),
        ("N_train=100",            {"N_train": 100}),
        ("N_train=300",            {"N_train": 300}),
    ]
    results = []

    for i, (name, overrides) in enumerate(specs, start=1):
        cfg = replace(base_cfg, **overrides)
        cfg.maxEpochs = 250
        cfg.patience = 50
        cfg.N_test = 100
        cfg.quick = False
        cfg.saveResults = False
        cfg.refresh_grids()

        rng = np.random.default_rng(cfg.seed)   # 每个配置从同一随机流开始
        torch.manual_seed(cfg.seed)

        print(f"\n  [{i}/{len(specs)}] {name} (m={cfg.m}, p={cfg.p}, N_train={cfg.N_train})")
        data = make_dataset(cfg, rng)
        model, hist = train_deeponet(cfg, data, rng)
        Rg = evaluate_split(model, data["te"], cfg)

        results.append({
            "name": name, "m": cfg.m, "p": cfg.p, "Ntr": cfg.N_train,
            "val": hist["bestVal"], "testMean": Rg["mean"],
            "testMedian": Rg["median"], "testP95": Rg["p95"],
            "epochs": hist["epochsRun"], "wall": hist["wallTotal"],
        })
        print(f"      -> test rel-L2: 均值 {Rg['mean']:.3e} | "
              f"中位数 {Rg['median']:.3e} | P95 {Rg['p95']:.3e}")

    print("\n  ---- 消融汇总 ----")
    print(f"  {'config':<24s} {'m':>4s} {'p':>4s} {'N':>6s} | "
          f"{'mean':>10s} {'median':>10s} {'P95':>10s} | {'epoch':>6s} {'time(s)':>8s}")
    for r in results:
        print(f"  {r['name']:<24s} {r['m']:4d} {r['p']:4d} {r['Ntr']:6d} | "
              f"{r['testMean']:10.3e} {r['testMedian']:10.3e} {r['testP95']:10.3e} | "
              f"{r['epochs']:6d} {r['wall']:8.1f}")

    print("\n  解读提示：m 变小时误差上升 -> 传感器采样表示受限；\n"
          "            p 变化不大 -> 模型容量不是瓶颈；\n"
          "            N_train 变小时误差上升 -> 数据量是瓶颈。")
    return results


In [13]:
print("\n---- 7. 消融实验（m / p / N_train）----")
abl = run_ablation(cfg)
with open(DATA / "A1组-6.3-消融结果.csv", "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(abl[0].keys()))
    writer.writeheader()
    writer.writerows(abl)
print("\n  消融结果已保存: 数据/A1组-6.3-消融结果.csv")



---- 7. 消融实验（m / p / N_train）----

---- 7. 消融实验（m / p / N_train）----
  说明：为控制 CPU 时间，消融统一用 250 epoch + 早停、N_test=100。

  [1/7] baseline m32 p32 N1000 (m=32, p=32, N_train=1000)
  epoch    1 | trainMSE 2.808e-01 | val rel-L2 2.682e-01 | lr 1.0e-03 |    0.1 s
  epoch   25 | trainMSE 2.458e-03 | val rel-L2 5.428e-02 | lr 1.0e-03 |    2.9 s
  epoch   50 | trainMSE 1.329e-03 | val rel-L2 4.046e-02 | lr 1.0e-03 |    5.7 s
  epoch   75 | trainMSE 9.881e-04 | val rel-L2 3.423e-02 | lr 1.0e-03 |    8.7 s
  epoch  100 | trainMSE 9.320e-04 | val rel-L2 3.521e-02 | lr 1.0e-03 |   11.7 s
  epoch  125 | trainMSE 5.423e-04 | val rel-L2 3.152e-02 | lr 1.0e-03 |   14.9 s
  epoch  150 | trainMSE 5.444e-04 | val rel-L2 2.250e-02 | lr 1.0e-03 |   17.9 s
  epoch  175 | trainMSE 1.095e-03 | val rel-L2 2.971e-02 | lr 1.0e-03 |   21.0 s
  epoch  200 | trainMSE 3.927e-04 | val rel-L2 2.067e-02 | lr 1.0e-03 |   24.1 s
  epoch  225 | trainMSE 4.525e-04 | val rel-L2 2.071e-02 | lr 1.0e-03 |   27.4 s
  epoch  250

## 结果解释与本节结论

主实验在验证集第 848 epoch 取得最优，验证相对 $L^2$ 误差 $4.176\times10^{-3}$；第 948 epoch 触发早停。测试集（新函数）相对 $L^2$ 均值 $4.144\times10^{-3}$、中位数 $3.857\times10^{-3}$、P95 $7.131\times10^{-3}$、$H^1$ 均值 $9.257\times10^{-2}$。最近邻记忆基线均值 $1.523\times10^{-1}$，新函数误差约为其 $1/37$，说明模型不是简单记忆训练函数。

分布外结果：幅值 OOD 约放大 $2.1$ 倍，频率 OOD 约放大 $28.8$ 倍；算子学习不能外推到训练分布之外。成本上，网络单次推理 $0.34$ ms（本次复跑 $0.307$ ms），谱求解器 $0.036$ ms（复跑 $0.040$ ms），二阶差分 $0.049$ ms（复跑 $0.046$ ms）；网络推理并不比传统求解器快，优势在“一次训练、批量新函数”的摊销场景。

消融显示：$m$ 减小和 $N_{\rm train}$ 减小会显著推高误差，$p=16/64$ 变化不大——传感器采样与数据量是瓶颈，模型容量不是。上述结论只适用于本节的一维、周期、线性算子与合成数据。

In [14]:
# ---------------- 结果落盘 ----------------
if cfg.saveResults:
    np.savez(
        DATA / "A1组-6.3-主实验数据.npz",
        ygrid=cfg.ygrid,
        trainMSE=np.asarray(hist["trainMSE"]),
        valRel=np.asarray(hist["valRel"]),
        bestEpoch=hist["bestEpoch"], bestVal=hist["bestVal"],
        wallTotal=hist["wallTotal"], epochsRun=hist["epochsRun"],
        eL2_test=Rg["eL2"], eL2_test_random=Rr["eL2"],
        eL2_train=Rt["eL2"], eL2_memo=Rm["eL2"],
        eL2_ood_amp=Ro["eL2"], eL2_ood_freq=Rf["eL2"],
        test_mean=Rg["mean"], test_median=Rg["median"], test_p95=Rg["p95"],
        test_maxAbs=Rg["maxAbs"], test_meanH1=Rg["meanH1"],
        memo_mean=Rm["mean"], oodAmp_mean=Ro["mean"], oodFreq_mean=Rf["mean"],
        t_net=t_net, t_spec=t_spec, t_fd=t_fd,
    )
    print("结果已保存: 数据/A1组-6.3-主实验数据.npz")

hw = hw_info()
summary = []
summary.append("# A1组-6.3 主实验汇总（实测）\n")
summary.append(f"- 生成时间：{time.strftime('%Y-%m-%d %H:%M:%S')}")
summary.append("- 代码：`A1组-实验-6.3.ipynb`")
summary.append(f"- 硬件：{hw['os']} | {hw['cpu']} | {hw['cores']} 逻辑核")
summary.append(f"- 软件：{hw['rel']} | NumPy {np.__version__}")
summary.append("")
summary.append("## 实验设定\n")
summary.append("- 算子：$G:u\\mapsto w$，$(-w''+w)=u$，$y\\in[0,1]$，周期边界。")
summary.append(f"- 训练 / 验证 / 测试：{cfg.N_train} / {cfg.N_val} / {cfg.N_test} 个独立输入函数，索引互不相交。")
summary.append(f"- 传感器点 $m={cfg.m}$，输出网格 $M={cfg.M}$；branch/trunk 潜在维数 $p={cfg.p}$，隐层 {cfg.hidden}。")
summary.append(f"- 优化：Adam，lr={cfg.lr:g}，batch={cfg.batchSize}，上限 {cfg.maxEpochs} epoch，早停耐心 {cfg.patience}。")
summary.append(f"- 数据生成：{data['nCalls']} 次求解器调用，墙钟 {gen_wall:.2f} s。")
summary.append("")
summary.append("## 训练\n")
summary.append(f"- 实际运行 {hist['epochsRun']} epoch，早停={hist['stoppedEarly']}；最优 epoch={hist['bestEpoch']}。")
summary.append(f"- 最优验证集相对 $L^2$ 误差：{hist['bestVal']:.3e}。")
summary.append(f"- 训练总墙钟：{hist['wallTotal']:.2f} s。")
summary.append("")
summary.append("## 测试指标（相对 $L^2$）\n")
summary.append("| 数据集 | 均值 | 中位数 | P95 | 最大 | 最大绝对误差 | H1 均值 |")
summary.append("|---|---:|---:|---:|---:|---:|---:|")
for nm, R in (("测试集（新函数，64 网格点）", Rg), ("测试集（新函数，16 随机坐标）", Rr),
              ("训练函数（仅参考）", Rt), ("最近邻记忆基线", Rm),
              (f"OOD 幅值 ×{cfg.oodAmpScale:g}", Ro), (f"OOD 频率 K={cfg.Kood}", Rf)):
    h1 = "—" if np.isnan(R["meanH1"]) else f"{R['meanH1']:.3e}"
    summary.append(f"| {nm} | {R['mean']:.3e} | {R['median']:.3e} | {R['p95']:.3e} | {R['maxRel']:.3e} | {R['maxAbs']:.3e} | {h1} |")
summary.append("")
summary.append(f"- 求解器自身相对解析解误差：均值 {np.mean(e_sol):.2e}，最大 {np.max(e_sol):.2e}。")
summary.append("")
summary.append("## 成本\n")
summary.append(f"- 网络单次推理：{t_net * 1e3:.3f} ms")
summary.append(f"- 谱求解器单次：{t_spec * 1e3:.3f} ms（自身相对误差 {e_spec:.1e}）")
summary.append(f"- 二阶差分求解器单次：{t_fd * 1e3:.3f} ms（自身相对误差 {e_fd:.1e}）")
summary.append(f"- 摊销（对谱求解器）：{msg_spec}")
summary.append(f"- 摊销（对差分求解器）：{msg_fd}")
summary.append("")
summary.append("## 判据\n")
summary.append(f"- 新函数误差 / 记忆基线误差 = {ratio:.3f}")
summary.append("- 结论：" + ("新输入函数上的误差显著低于“只记住训练函数”的基线，支持“学到了算子”。" if Rg["mean"] < 0.5 * Rm["mean"] else "与记忆基线差距不足，不能宣称学到了算子。"))
summary.append(f"- 新函数（网格）{Rg['mean']:.3e} vs 同函数新坐标 {Rr['mean']:.3e}。")
summary.append("")
summary.append("## 消融\n")
summary.append("| 配置 | m | p | N_train | 验证 rel-L2 | 测试均值 | 中位数 | P95 | epoch | 墙钟/s |")
summary.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
for r in abl:
    summary.append(f"| {r['name']} | {r['m']} | {r['p']} | {r['Ntr']} | {r['val']:.3e} | {r['testMean']:.3e} | {r['testMedian']:.3e} | {r['testP95']:.3e} | {r['epochs']} | {r['wall']:.1f} |")
summary.append("")
summary.append("> 消融墙钟为本次复跑值：满 250 epoch 的组约 30–34 s，N_train=100 组约 2 s，N_train=300 组约 10 s；正文表 6.3-14 为正式运行值（约 16–19 s/组）。两者指标一致。")
summary.append("")
summary.append("> 注：正式长任务日志见 `辅助文件/A1组-6.3-实验日志.md`；其中 807.83 s 含约 750 s 系统挂起，无干扰运行约 64.5 s。本 Notebook 复跑耗时以本次输出为准。")
(DATA / "A1组-6.3-主实验汇总.md").write_text("\n".join(summary) + "\n", encoding="utf-8")
print("结果已保存: 数据/A1组-6.3-主实验汇总.md")


结果已保存: 数据/A1组-6.3-主实验数据.npz
结果已保存: 数据/A1组-6.3-主实验汇总.md


## 6.3 实际运行记录

- **运行日期：** 2026-10-27（本 Notebook 交付前复跑）。
- **环境：** Windows；Python 3.12.15（Anaconda）；PyTorch 2.14.1+cpu；NumPy 2.5.4；Matplotlib 3.11.2；CPU，4 线程。
- **随机性：** 数据生成与训练种子均为 2024；训练/验证/测试索引固定。
- **正式长任务：** 原始脚本记录的总墙钟 807.83 s，其中约 750 s 为系统挂起；无干扰运行约 64.5 s。本 Notebook 复跑的主训练墙钟 83.19 s、消融 7 组共约 174 s；指标与正文表 6.3-1、6.3-2 和消融表一致。
- **检查点：** `辅助文件/A1组-6.3-模型检查点.pt`（二进制权重，无法转成文本；保留用于复现评估，理由见 `辅助文件/A1组-6.3-说明.md`）。
- **路径：** 所有读写均为相对路径；代码只写入 `图片/`、`数据/`、`辅助文件/` 三个既有目录，不创建新文件夹。
- **一致性：** 复跑指标与正文表 6.3-1、6.3-2 和消融表一致；若硬件不同，墙钟会变化，误差指标应在同一数量级。